In [0]:
%sql
SELECT 
full_name as nome,sum(total) as fraturamento_total
, count(actual_category) OVER (partition by full_name) as categoria
, fraturamento_total/count(sale_date) as ticket_medio

FROM fatos_vendas 
LEFT JOIN dim_clientes  ON fatos_vendas.id_client = dim_clientes.code 
left join lh_nautical.gold_lh_nautical.dim_produtos on fatos_vendas.id_product = dim_produtos.code
GROUP BY full_name,actual_category

In [0]:
%sql
use catalog lh_nautical;
use schema gold_lh_nautical;

with calculo_ticket_medio as (
SELECT 
full_name as nome
, sum(total) as fraturamento_total
, sum(qtd) as quantidades_vendidas
, actual_category
, count(actual_category) OVER (partition by full_name) as diversidade -- Usei um count over para contar a quantidade de categorias por cliente e numa outra CTE eu filtrei
, fraturamento_total/count(sale_date) as ticket_medio

FROM fatos_vendas 
LEFT JOIN dim_clientes  ON fatos_vendas.id_client = dim_clientes.code 
left join lh_nautical.gold_lh_nautical.dim_produtos on fatos_vendas.id_product = dim_produtos.code
GROUP BY full_name,actual_category
),

maior_ticket_medio_por_cliente as (
SELECT 
nome
, sum(ticket_medio) as ticket_medio
FROM calculo_ticket_medio 
where diversidade>=3
GROUP BY nome
)
, ticket_ranked as (
    select *,
    RANK() OVER (ORDER BY ticket_medio DESC) as rk -- Usei um rank over para fazer o ranking do ticket medio por cliente, e depois usei essa informacao em um filtro na CTE final
    FROM maior_ticket_medio_por_cliente
    qualify rk<=10
)
select 
    actual_category
    ,sum(quantidades_vendidas) as total
from calculo_ticket_medio
where nome in ( select nome from ticket_ranked )
group by actual_category
order by total desc
-- limit 1
-- grouped as (
--     SELECT 
-- nome
-- , quantidades_vendidas
-- ,sum(fraturamento_total) as fraturamento_total
-- , diversidade
-- , sum(ticket_medio) as ticket_medio

-- FROM ticket_medio 
-- where diversidade>=3
-- GROUP BY nome,diversidade
-- )

-- SELECT 
-- *
-- , RANK() OVER (ORDER BY ticket_medio DESC) as rk
-- FROM grouped



In [0]:
%sql
use catalog lh_nautical;
use schema gold_lh_nautical;

SELECT full_name as nome,sum(total) as fraturamento_total, fraturamento_total/count(sale_date) as ticket_medio FROM fatos_vendas LEFT JOIN dim_clientes  ON fatos_vendas.id_client = dim_clientes.code GROUP BY id_client,full_name